In [1]:
# Cell 1: import required library
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
# Cell 2: Load datase & setup
# Paths
AUDIT_DIR = Path("../reports/audit")
RAW_DATA_DIR = Path("../data/raw/data1a")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok= True)
FIGURES_DIR = Path("../reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok= True)

# Load inventory data
df = pd.read_csv(AUDIT_DIR / "dataset_inventory.csv")

# Ensure labels are binary for splitting
df["target"] = (df["label"] == "damaged").astype(int)

print(f"Loaded {len(df)} images.")
print(f"Unique groups: {df['group_id'].nunique()}")

Loaded 2300 images.
Unique groups: 2278


In [3]:
# Cell 3: Stratified Group-Aware Split
# We want 70% train, 15% val, 15% test. 
# StratifiedGroupKFold doesn't do 3-way directly, so we split 70/30 first, then 50/50.
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)

# Get the first split as 70% train / 30% temp
# (10 folds, so taking 7 folds for train)
splits = list(sgkf.split(df, df["target"], groups=df["group_id"]))
train_idx, temp_idx = splits[0] # This gives us 70% train, 30% temp

df_train = df.iloc[train_idx].copy()
df_temp = df.iloc[temp_idx].copy()


unique_groups = df["group_id"].unique()
group_labels = df.groupby("group_id")["target"].mean().round().astype(int).to_dict()

np.random.seed(42)
damaged_groups = [g for g in unique_groups if group_labels[g] == 1]
whole_groups = [g for g in unique_groups if group_labels[g] == 0]

np.random.shuffle(damaged_groups)
np.random.shuffle(whole_groups)

# Calculate split indices for each class to ensure 50/50
# We have ~1150 damaged and ~1150 whole
# We want 70% train, 15% val, 15% test for BOTH classes.

def split_groups(groups, train_ratio=0.70, val_ratio=0.15):
    n = len(groups)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    return groups[:train_end], groups[train_end:val_end], groups[val_end:]

d_train, d_val, d_test = split_groups(damaged_groups)
w_train, w_val, w_test = split_groups(whole_groups)

train_groups = d_train + w_train
val_groups = d_val + w_val
test_groups = d_test + w_test

df["split_final"] = "unassigned"
df.loc[df["group_id"].isin(train_groups), "split_final"] = "train"
df.loc[df["group_id"].isin(val_groups), "split_final"] = "val"
df.loc[df["group_id"].isin(test_groups), "split_final"] = "test"

print("--- Split Sizes ---")
print(df["split_final"].value_counts())
print("\n--- Class Balance per Split ---")
print(pd.crosstab(df["split_final"], df["label"]))

--- Split Sizes ---
split_final
train    1611
test      346
val       343
Name: count, dtype: int64

--- Class Balance per Split ---
label        damaged  whole
split_final                
test             172    174
train            807    804
val              171    172


In [4]:
# Cell 4: Verify, Stats, and Save
g = df.groupby("group_id")
print(f"Multi-image groups: {(g.size() > 1).sum()} | Largest group: {g.size().max()}")
print(f"Label-conflict groups (same image, different labels): {(g['label'].nunique() > 1).sum()}")

leak_check = df.groupby("group_id")["split_final"].nunique()
leaks = leak_check[leak_check > 1]

if len(leaks) == 0:
    print("SUCCESS: No detected duplicate group spans across multiple splits.")
else:
    print(f"WARNING: {len(leaks)} groups span across splits!")

# Save the split files
df[df["split_final"] == "train"].to_csv(PROCESSED_DIR / "train_split.csv", index=False)
df[df["split_final"] == "val"].to_csv(PROCESSED_DIR / "val_split.csv", index=False)
df[df["split_final"] == "test"].to_csv(PROCESSED_DIR / "test_split.csv", index=False)

print("\Saved train_split.csv, val_split.csv, and test_split.csv to data/processed/")

Multi-image groups: 22 | Largest group: 2
Label-conflict groups (same image, different labels): 0
SUCCESS: No detected duplicate group spans across multiple splits.
\Saved train_split.csv, val_split.csv, and test_split.csv to data/processed/


In [5]:
# Cell 5: Define the Transforms
import torchvision.transforms as T 
from torchvision.transforms import v2
# 1. Train Transforms (Aggressive Anti-Shortcut)
train_transforms = T.Compose([
    T.Resize((256, 256)), # Standardize resolution first
    T.RandomCrop(224),    # Mild crop
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=10),
    # Color Augmentation
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    # ANTI-SHORTCUT AUGMENTATIONS
    T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.3),
    # ToTensor and Normalization (ImageNet stats)
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Validation / Test Transforms (Clean & Deterministic)
val_test_transforms = T.Compose([
    T.Resize((256, 256)),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


print("Transforms defined.")

Transforms defined.


In [6]:
# Cell 6: Custom Dataset Class
class CarDamageDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        """
        Args: DataFrame contining 'image_path' and 'target'
        root_dir (Path): Root directory where images are stored
        transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.df = dataframe.reset_index(drop= True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Get image path and label
        img_rel_path = self.df.loc[idx, "image_path"]
        img_path = self.root_dir / img_rel_path
        label = self.df.loc[idx, "target"]

        # Open image and force RGB (fixes grayscale)
        image = Image.open(img_path).convert("RGB")

        # Apply transforms
        if self.transform:
            image = self.transform(image)

        return image, label

print("Dataset class defined.")

Dataset class defined.


In [7]:
# Cell 7: Instantiate Datasets and DataLoaders
BATCH_SIZE = 32

# 1. Load the split CSVs 
df_train = pd.read_csv(PROCESSED_DIR / "train_split.csv")
df_val = pd.read_csv(PROCESSED_DIR / "val_split.csv")
df_test = pd.read_csv(PROCESSED_DIR / "test_split.csv")

# 2. Create Datasets
train_dataset = CarDamageDataset(df_train, RAW_DATA_DIR, transform=train_transforms)
val_dataset = CarDamageDataset(df_val, RAW_DATA_DIR, transform=val_test_transforms)
test_dataset = CarDamageDataset(df_test, RAW_DATA_DIR, transform=val_test_transforms)

# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

# 4. Test a single batch
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}")
print(f"Label shape: {labels.shape}")
print(f"Labels in batch: {labels.numpy()}")

Train batches: 51
Val batches: 11
Test batches: 11

Batch shape: torch.Size([32, 3, 224, 224])
Label shape: torch.Size([32])
Labels in batch: [0 0 1 0 0 1 0 1 0 1 1 1 1 0 1 0 1 0 0 1 0 1 0 0 0 0 0 1 1 0 1 1]


## Preprocessing & Augmentation Summary

**1. Duplicate groups**
- 2300 images grouped by exact (MD5) and near-duplicate (pHash, Hamming <= 5) matches: 2278 unique groups (22 groups with more than one image, largest group = 2 images).
- Groups with conflicting labels: 0. (Action: None needed).

**2. Group-aware split** (seed 42)
- The original train/validation split was discarded because of cross-split near-duplicates.
- Final sizes: Train 1611 (70.0%), Val 343 (15.0%), Test 346 (15.0%).
- Class balance (damaged/whole): Train 807/804, Val 171/172, Test 172/174.
- Verified: no detected duplicate group spans more than one split. (Near-duplicates below the pHash threshold may remain).
- Validation is used for model selection; the test set is used once, for the final chosen model.

**3. Transforms**
- All images converted to RGB; resized to 224x224 with the same method for both classes; ImageNet normalization.
- Train: RandomCrop (mild), RandomHorizontalFlip, RandomRotation, ColorJitter, GaussianBlur.
- Val/Test: Deterministic resize + normalization only.
- Rationale: Reduce reliance on resolution/compression cues found in EDA. This reduces, but does not prove removal of, the source shortcut; to be tested with Grad-CAM.

**4. Pipeline**
- `CarDamageDataset` + DataLoaders, batch size 32.
- Train/val/test batches: 51/11/11. Batch tensor shape [32, 3, 224, 224].

**5. Limitations**
- pHash threshold of 5 may miss some near-duplicates.
- Source/scene-style differences remain after resizing.
- No out-of-distribution set yet.